In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/dev.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['dev'][0])
print(len(data['dev']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 5, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 1, 'time_since_start': 0.4363888888888889, 'time_since_last_event': 0.4363888888888889}, {'idx_event': 3, 'type_event': 5, 'time_since_start': 1.1122222222222222, 'time_since_last_event': 0.6758333333333333}, {'idx_event': 4, 'type_event': 0, 'time_since_start': 1.4241666666666666, 'time_since_last_event': 0.31194444444444436}, {'idx_event': 5, 'type_event': 5, 'time_since_start': 1.8616666666666666, 'time_since_last_event': 0.4375}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 2.2730555555555556, 'time_since_last_event': 0.411388888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 3.278888888888889, 'time_since_last_event': 1.0058333333333334}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 3.4175, 'time_since_last_event': 0.13861111111111102}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 3.460277777777778, 'time_since_last_eve

In [4]:
from src.data.sequence import Sequence,EventSequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return EventSequence(
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
data.keys()

dict_keys(['dim_process', 'dev'])

In [6]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["dev"]]

In [7]:
from src.data.batch import Batch

In [8]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [9]:
for batch in loader:
    print(batch.keys())
    break

['arrival_times', 'inter_times', 'non_pad_mask', 'type_seq', 't_start', 't_end']


In [10]:
batch.type_seq[2]

tensor([8, 0, 5, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 0, 5, 3,
        8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3])

In [11]:
batch.type_seq

tensor([[8, 3, 8,  ..., 3, 8, 3],
        [8, 3, 8,  ..., 3, 9, 9],
        [8, 0, 5,  ..., 3, 8, 3],
        ...,
        [8, 3, 8,  ..., 3, 8, 1],
        [8, 3, 8,  ..., 0, 5, 3],
        [8, 3, 8,  ..., 1, 9, 9]])

In [12]:
batch.arrival_times

tensor([[ 0.0000,  0.0942,  0.1244,  ...,  6.3375,  6.3942,  6.6978],
        [ 0.0000,  0.1697,  0.2739,  ...,  7.5436,  0.0000,  0.0000],
        [ 0.0000,  0.2614,  0.9053,  ..., 10.4017, 10.4400, 10.4872],
        ...,
        [ 0.0000,  0.0531,  0.0656,  ...,  4.2178,  4.4281,  4.8517],
        [ 0.0000,  0.1492,  0.4369,  ...,  8.4383,  8.4700,  8.7142],
        [ 0.0000,  0.1875,  0.2178,  ...,  8.0931,  0.0000,  0.0000]])

In [13]:
batch.inter_times

tensor([[0.0000, 0.0942, 0.0303,  ..., 0.2469, 0.0567, 0.3036],
        [0.0000, 0.1697, 0.1042,  ..., 0.1228, 0.0000, 0.0000],
        [0.0000, 0.2614, 0.6439,  ..., 0.0853, 0.0383, 0.0472],
        ...,
        [0.0000, 0.0531, 0.0125,  ..., 0.0825, 0.2103, 0.4236],
        [0.0000, 0.1492, 0.2878,  ..., 0.0436, 0.0317, 0.2442],
        [0.0000, 0.1875, 0.0303,  ..., 0.4517, 0.0000, 0.0000]])

In [14]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
args.dataset = "taxi"
base_dir = f"data/{args.dataset}"

In [15]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

In [16]:
for batch in train_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    print(f"type_event: {batch.type_seq}")
    print(f"non_pad_mask: {batch.non_pad_mask}")
    print(f"t_start: {batch.t_start}")
    print(f"t_end: {batch.t_end}")  
    break

arival_times: tensor([[ 0.0000,  0.2761,  0.6211,  ...,  5.5044,  7.8169,  7.9933],
        [ 0.0000,  0.6569,  0.8319,  ...,  8.9153,  0.0000,  0.0000],
        [ 0.0000,  0.0617,  0.2186,  ..., 11.5661, 11.6439, 11.6925],
        ...,
        [ 0.0000,  0.2456,  0.2600,  ...,  6.8425,  0.0000,  0.0000],
        [ 0.0000,  0.1603,  0.1753,  ...,  6.9289,  0.0000,  0.0000],
        [ 0.0000,  0.1897,  0.2586,  ...,  9.5119, 12.2681, 12.7225]])
inter_times: tensor([[0.0000, 0.2761, 0.3450,  ..., 0.2994, 2.3125, 0.1764],
        [0.0000, 0.6569, 0.1750,  ..., 0.4703, 0.0000, 0.0000],
        [0.0000, 0.0617, 0.1569,  ..., 0.2719, 0.0778, 0.0486],
        ...,
        [0.0000, 0.2456, 0.0144,  ..., 1.1364, 0.0000, 0.0000],
        [0.0000, 0.1603, 0.0150,  ..., 0.1300, 0.0000, 0.0000],
        [0.0000, 0.1897, 0.0689,  ..., 0.4894, 2.7561, 0.4544]])
type_event: tensor([[8, 1, 8,  ..., 3, 8, 1],
        [5, 3, 8,  ..., 3, 9, 9],
        [8, 3, 8,  ..., 3, 8, 3],
        ...,
        [8, 3,

In [17]:
batch[:,:-3]

EventBatch(
  arrival_times: [32, 35],
  inter_times: [32, 35],
  non_pad_mask: [32, 35],
  type_seq: [32, 35],
  t_start: [32],
  t_end: [32]
)

In [18]:
for batch in train_loader:
   print(
    batch.inter_times.max().item(),
    batch.inter_times.min().item(),
    batch.inter_times.mean().item()
)

   

5.721388816833496 0.0 0.21718817949295044
4.752500057220459 0.0 0.20207147300243378
4.097222328186035 0.0 0.2147034853696823
3.7125000953674316 0.0 0.20799799263477325
4.507777690887451 0.0 0.20207853615283966
4.869999885559082 0.0 0.23370341956615448
2.495833396911621 0.0 0.2135377675294876
3.257499933242798 0.0 0.2085585743188858
5.065833568572998 0.0 0.2231069654226303
3.9397222995758057 0.0 0.22613371908664703
5.111666679382324 0.0 0.20819193124771118
3.071666717529297 0.0 0.2065250426530838
2.8088889122009277 0.0 0.19937317073345184
3.888611078262329 0.0 0.21677882969379425
3.320833444595337 0.0 0.20837080478668213
3.008333444595337 0.0 0.21462468802928925
3.6875 0.0 0.19848890602588654
2.5402777194976807 0.0 0.20031295716762543
5.420555591583252 0.0 0.21507468819618225
5.524722099304199 0.0 0.22809073328971863
3.295555591583252 0.0 0.21929480135440826
2.8341667652130127 0.0 0.21014390885829926
2.434999942779541 0.0 0.2112787961959839
3.0805554389953613 0.0 0.2103661745786667
4.03

In [19]:
batch.inter_times[0]

tensor([0.0000, 0.2067, 0.0136, 0.1044, 0.0078, 0.1825, 0.0914, 0.2217, 0.0278,
        0.0575, 0.1444, 0.4794, 0.2619, 0.1297, 0.2403, 0.1147, 0.2322, 0.1900,
        0.0561, 0.2981, 0.8975, 0.1303, 0.1569, 0.3967, 0.0547, 0.1019, 0.0158,
        0.0589, 0.2411, 0.1544, 0.3164, 0.0781, 0.0603, 0.1286, 0.1342, 0.0547,
        1.7853, 0.1483])

In [20]:
batch.inter_times[0].max().item()

1.7852777242660522